# PV Yield Forecaster — reproducible analysis

Reproduces the README benchmark table and figures from NASA POWER data. Responses are cached in `data/cache/`, so re-running is offline after the first fetch.

Command-line equivalent: `python -m pvyield benchmark --out docs/benchmark_metrics.csv --figures docs/figures`

In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT / 'src'))

from pvyield.cities import CITIES
from pvyield.data import fetch_daily
from pvyield.evaluate import benchmark, evaluate_forecaster, feature_importance
from pvyield.physics import physics_yield
from pvyield.plots import (
    daily_yield_figure,
    feature_importance_figure,
    monthly_profile_figure,
    residuals_by_month_figure,
)

## 1. Benchmark: persistence vs climatology vs ML on the test window

In [ ]:
start, end, system_kw = '2019-01-01', '2024-12-31', 5.0
table = benchmark(['Faisalabad', 'Karachi', 'Quetta', 'Skardu'], start, end, system_kw)
display(table.style.format('{:.3f}'))

## 2. Faisalabad in detail

In [ ]:
weather = fetch_daily('Faisalabad', start, end)
lat = CITIES['Faisalabad']['lat']
daily = physics_yield(weather, lat=lat, system_kw=system_kw)
annual = daily.mean() * 365.25
print(f'Annualised yield: {annual:,.0f} kWh ({annual / system_kw:,.0f} kWh/kWp)')
monthly_profile_figure(daily).show()

In [ ]:
evaluation = evaluate_forecaster(weather, lat=lat, system_kw=system_kw)
name = evaluation.trained.model_name
print(f'{name}: {evaluation.trained.best_iteration} boosting rounds chosen on the validation window')
predictions = evaluation.predictions
daily_yield_figure(predictions[['actual', name, 'persistence']]).show()
feature_importance_figure(feature_importance(evaluation.trained)).show()
residuals_by_month_figure(predictions['actual'], predictions[name]).show()